<a href="https://colab.research.google.com/github/Sineth7077/project-aura/blob/main/notebooks/04_inference_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project Aura 2.0 - Notebook 04: Inference Engine

This notebook serves as the fully production-ready **Inference Engine** for Project Aura. Following the training phase in Notebook 03, we now load the serialized model weights to evaluate real-time performance and perform batch predictions.

### Objectives:
1. **Restore State**: Build/load the exact trained `EfficientNet-B0` architecture with CIFAR-10 mapping.
2. **Preprocessing Integrity**: Map inputs to target configurations (RGB, 224x224, normalized PyTorch Tensors).
3. **Perform Inference**: Conduct both single and high-efficiency batched predictions.
4. **Performance Benchmark**: Analyze inference speed, per-class accuracies, and save verified evaluation reports directly to Google Drive.

In [10]:
import os
import sys
import time
import json
import numpy as np
import torch
import torchvision
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import drive

# Mount Google Drive to locate checkpoints and export paths
try:
    drive.mount('/content/drive', force_remount=True)
    print("Google Drive mounted successfully.")
except Exception as e:
    print(f"Could not mount Google Drive (running locally or without credentials): {e}")

PROJECT_DIR = "/content/project-aura"
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Fallback architecture configuration if src module is absent
# Ensure PyTorch uses GPU if CUDA is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using runtime device: {device}")
print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")

Could not mount Google Drive (running locally or without credentials): Error: credential propagation was unsuccessful
Using runtime device: cpu
PyTorch version: 2.14.1+cpu
Torchvision version: 0.29.1+cpu


In [16]:

# SECTION 3 — Reconstruct and Load the Trained Model

from torchvision.models import efficientnet_b0
import torch.nn as nn
import os

CHECKPOINT_PATH = (
    "/content/drive/MyDrive/ProjectAura/checkpoints/"
    "efficientnet_b0_cifar10_best.pth"
)

def build_cifar10_model():
    """Reconstruct the EfficientNet-B0 CIFAR-10 architecture."""
    model = efficientnet_b0(weights=None)
    model.classifier[1] = nn.Linear(
        model.classifier[1].in_features,
        10
    )
    return model


# Verify checkpoint availability
if not os.path.isfile(CHECKPOINT_PATH):
    raise FileNotFoundError(
        f"Trained checkpoint not found: {CHECKPOINT_PATH}\n"
        "Mount Google Drive and verify the checkpoint location. "
        "Do not continue with randomly initialized weights."
    )

# Load checkpoint
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=True
)

# Extract model state dictionary
if "model_state_dict" in checkpoint:
    model_state = checkpoint["model_state_dict"]
elif "state_dict" in checkpoint:
    model_state = checkpoint["state_dict"]
else:
    model_state = checkpoint

# Handle DataParallel prefixes
model_state = {
    key.removeprefix("module."): value
    for key, value in model_state.items()
}

# Reconstruct architecture and restore trained weights
model = build_cifar10_model()
model.load_state_dict(model_state, strict=True)

model = model.to(device)
model.eval()

# Confirm successful loading
print("=" * 50)
print("PROJECT AURA — MODEL LOADING REPORT")
print("=" * 50)
print("Architecture: EfficientNet-B0")
print("Dataset: CIFAR-10")
print("Classes: 10")
print("Checkpoint:", CHECKPOINT_PATH)
print("Device:", device)
print("Evaluation mode:", not model.training)
print("Trained weights loaded: SUCCESS")
print("=" * 50)


FileNotFoundError: Trained checkpoint not found: /content/drive/MyDrive/ProjectAura/checkpoints/efficientnet_b0_cifar10_best.pth
Mount Google Drive and verify the checkpoint location. Do not continue with randomly initialized weights.

In [18]:

# PROJECT AURA — CHECKPOINT DIAGNOSTIC

import os
from google.colab import drive

# Mount Google Drive
drive.mount("/content/drive", force_remount=True)

ROOT = "/content/drive/MyDrive"

print("\n1. GOOGLE DRIVE STATUS")
print("MyDrive exists:", os.path.exists(ROOT))

if not os.path.exists(ROOT):
    raise RuntimeError("Google Drive is not mounted correctly.")

print("\n2. MYDRIVE FOLDERS")
for item in os.listdir(ROOT):
    print(item)

print("\n3. SEARCHING FOR PROJECT AURA CHECKPOINTS")

found = []

for current_root, dirs, files in os.walk(ROOT):
    for filename in files:
        if filename.endswith(".pth"):
            path = os.path.join(current_root, filename)
            found.append(path)
            print("FOUND:", path)

print("\n4. CHECKPOINT SEARCH SUMMARY")
if found:
    print(f"Found {len(found)} PyTorch checkpoint file(s).")
else:
    print("NO .pth CHECKPOINT FILES FOUND IN MYDRIVE.")

print("\n5. EXPECTED CHECKPOINT")
expected = os.path.join(
    ROOT,
    "ProjectAura",
    "checkpoints",
    "efficientnet_b0_cifar10_best.pth"
)
print("Expected path:", expected)
print("Exists:", os.path.isfile(expected))


MessageError: Error: credential propagation was unsuccessful

In [14]:

# Load genuine CIFAR-10 test images

from torchvision.datasets import CIFAR10

cifar_test = CIFAR10(
    root="/content/project-aura-verified/data",
    train=False,
    download=True
)

# Select five real test images
test_images = [
    cifar_test[i][0] for i in range(5)
]

print("Loaded genuine CIFAR-10 test images:", len(test_images))

# Display the first image
plt.figure(figsize=(5, 5))
plt.imshow(test_images[0])
plt.title(f"Actual class: {cifar10_classes[cifar_test[0][1]]}")
plt.axis("off")
plt.show()

# Genuine single-image inference
single_result = predict_single(test_images[0])

print(json.dumps(single_result, indent=2))

# Genuine batch inference
batch_results, elapsed_ms = predict_batch(test_images)

for i, result in enumerate(batch_results):
    actual_class = cifar10_classes[cifar_test[i][1]]

    print(
        f"Image {i + 1}: "
        f"Actual={actual_class}, "
        f"Predicted={result['predicted_class']}, "
        f"Confidence={result['confidence']:.2%}"
    )

print(f"Total latency: {elapsed_ms:.2f} ms")


  4%|▍         | 7.60M/170M [00:48<17:18, 157kB/s]


KeyboardInterrupt: 

In [ ]:
# Section 4 & 5: Preprocessing and Class Labels
from torchvision import transforms

cifar10_classes = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

# Unified preprocessing pipeline matching the model training phase
inference_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def preprocess_image(image_input):
    """Accepts image paths or PIL Images, converts to RGB, transforms and adds batch dim."""
    if isinstance(image_input, str):
        img = Image.open(image_input).convert("RGB")
    elif isinstance(image_input, Image.Image):
        img = image_input.convert("RGB")
    elif isinstance(image_input, np.ndarray):
        img = Image.fromarray(image_input).convert("RGB")
    else:
        raise ValueError("Unsupported image format. Must be file path, PIL Image, or Numpy array.")

    tensor = inference_transforms(img)
    return tensor.unsqueeze(0).to(device)  # Add batch dimension and place on target device

In [ ]:
# Section 6: Reusable Single-Image Inference

def predict_single(image_input):
    start_time = time.time()
    tensor = preprocess_image(image_input)

    with torch.inference_mode():
        outputs = model(tensor)
        probabilities = torch.softmax(outputs, dim=1)[0]

    latency_ms = (time.time() - start_time) * 1000
    top_probs, top_classes = torch.topk(probabilities, 5)

    predictions = [
        {"class": cifar10_classes[cls_idx], "confidence": float(prob)}
        for prob, cls_idx in zip(top_probs, top_classes)
    ]

    return {
        "predicted_class": predictions[0]["class"],
        "confidence": predictions[0]["confidence"],
        "top_5": predictions,
        "latency_ms": latency_ms
    }

# Run on test PIL Image
test_pil = Image.new("RGB", (256, 256), (120, 200, 80))
result = predict_single(test_pil)
print("Single Inference Output:\n", json.dumps(result, indent=2))

In [ ]:
# Section 7 & 10: Batched Inference and Results Export

def predict_batch(images):
    """Runs efficient batch inference on a list of images."""
    if not images:
        return []

    start_time = time.time()
    tensors = [inference_transforms(Image.fromarray(img).convert("RGB")) if isinstance(img, np.ndarray)
               else inference_transforms(img.convert("RGB")) for img in images]

    batch_tensor = torch.stack(tensors).to(device)

    with torch.inference_mode():
        outputs = model(batch_tensor)
        probabilities = torch.softmax(outputs, dim=1)

    latency_ms = (time.time() - start_time) * 1000

    results = []
    for i in range(len(images)):
        probs = probabilities[i]
        top_prob, top_idx = torch.max(probs, dim=0)
        results.append({
            "predicted_class": cifar10_classes[top_idx.item()],
            "confidence": float(top_prob.item())
        })

    return results, latency_ms

# Create a mock batch
batch_samples = [Image.new("RGB", (150, 150), (np.random.randint(0,255), np.random.randint(0,255), np.random.randint(0,255))) for _ in range(5)]
batch_results, elapsed_ms = predict_batch(batch_samples)
print(f"Processed {len(batch_samples)} images in {elapsed_ms:.2f} ms ({elapsed_ms/len(batch_samples):.2f} ms/image)")

# Export to Google Drive
save_dir = "/content/drive/MyDrive/ProjectAura/inference"
os.makedirs(save_dir, exist_ok=True)
with open(os.path.join(save_dir, "inference_results.json"), "w") as f:
    json.dump({"batch_inference": batch_results, "average_latency_ms": elapsed_ms/len(batch_samples)}, f, indent=2)

In [ ]:
# Section 12 & 13: Validation Suite and Memory Cleanup
validation_results = {
    "Model Loaded successfully": model is not None,
    "Evaluation Mode Confirmed": not model.training,
    "GPU Acceleration Active": next(model.parameters()).is_cuda if torch.cuda.is_available() else True,
    "Export Directory Ready": os.path.exists(save_dir)
}

print("=== FINAL SYSTEM VALIDATION REPORT ===")
for test, status in validation_results.items():
    print(f"{test:<35}: {'[PASS]' if status else '[FAIL]'}")

# Garbage collection & free memory
torch.cuda.empty_cache()
print("\nGPU VRAM Cache Flushed. Notebook complete.")